# Crypto Transaction Fraud Detection — Elliptic Bitcoin Dataset

An end-to-end pipeline for detecting illicit Bitcoin transactions using the
[Elliptic Data Set](https://www.kaggle.com/datasets/ellipticco/elliptic-data-set):
~203k transactions, 166 anonymized features each, connected in a transaction
graph, with a **licit / illicit / unknown** label on each node.

**Why this dataset is a good fraud-detection testbed, and why the pipeline is
built the way it is:**

- **Severe class imbalance**: illicit transactions are a small minority (~2%
  of *labeled* transactions). Accuracy is a useless metric here — a model
  that predicts "licit" for everything scores >90% accuracy while catching
  zero fraud. We optimize for **Precision-Recall AUC** and **recall**
  instead (see Section 7 for why).
- **Time structure**: the data spans 49 time steps, and illicit activity
  patterns *drift* over time (new laundering techniques, exchange takedowns,
  etc.). Money laundering in the real world evolves to evade whatever
  detector is currently deployed, so we **split by time step** (train on
  early steps, test on later ones) rather than a random split, to honestly
  simulate deploying a model into the future.
- **Graph structure**: transactions form a payment graph. Illicit actors
  often exhibit distinctive graph topology (fan-out to many wallets, being
  fed by other illicit nodes, unusual centrality), so we engineer graph
  features (degree, PageRank) on top of the 166 given features.

This notebook is written to run top-to-bottom with no manual intervention,
aside from the Kaggle dataset download step (Section 2), which requires a
Kaggle API credential to be configured on the machine running this notebook.


## 1. Setup & Imports

We pin `random_state=42` everywhere a stochastic algorithm is used, so the
whole notebook is reproducible.


In [ ]:
# --- Core ---
import os
import json
import warnings
import pickle
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

# --- Viz ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Graph ---
import networkx as nx

# --- ML ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_auc_score,
)
from xgboost import XGBClassifier

# --- Imbalance handling ---
from imblearn.over_sampling import SMOTE

# --- Explainability (optional, guarded import) ---
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("shap not installed — feature importance will fall back to "
          "built-in model importances. Run `pip install shap` for SHAP plots.")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

DATA_DIR = Path("data/elliptic_bitcoin_dataset")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print("Environment ready.")


## 2. Dataset Download

We try `kagglehub` first (simplest, no manual auth file needed if you're in
a Kaggle-aware environment or have `~/.kaggle/kaggle.json` set up). If that
fails, we fall back to the official `kaggle` CLI, and if *that* isn't
configured either, we print manual-download instructions and stop cleanly
rather than crashing later cells with a confusing `FileNotFoundError`.

**Manual fallback**: if you don't have a Kaggle API key, download the
dataset directly from
https://www.kaggle.com/datasets/ellipticco/elliptic-data-set, unzip it, and
place the 3 CSVs in `data/elliptic_bitcoin_dataset/`.


In [ ]:
def download_elliptic_dataset(dest_dir: Path) -> Path:
    '''Download the Elliptic dataset via kagglehub, falling back to the
    kaggle CLI, and finally to a manual-download instruction if neither
    is authenticated. Returns the directory containing the 3 CSV files.
    '''
    dest_dir.mkdir(parents=True, exist_ok=True)

    expected_files = {
        "elliptic_txs_features.csv",
        "elliptic_txs_edgelist.csv",
        "elliptic_txs_classes.csv",
    }

    def _has_all_files(d: Path) -> bool:
        return expected_files.issubset({p.name for p in d.rglob("*.csv")})

    if _has_all_files(dest_dir):
        print(f"Found existing dataset files in {dest_dir}, skipping download.")
        return dest_dir

    # --- Attempt 1: kagglehub ---
    try:
        import kagglehub
        path = kagglehub.dataset_download("ellipticco/elliptic-data-set")
        path = Path(path)
        print(f"Downloaded via kagglehub to {path}")
        return path
    except Exception as e:
        print(f"kagglehub download failed ({e}); trying kaggle CLI...")

    # --- Attempt 2: kaggle CLI ---
    try:
        import kaggle  # noqa: F401  (import triggers auth check)
        os.system(
            f"kaggle datasets download -d ellipticco/elliptic-data-set "
            f"-p {dest_dir} --unzip"
        )
        if _has_all_files(dest_dir):
            print(f"Downloaded via kaggle CLI to {dest_dir}")
            return dest_dir
        raise FileNotFoundError("kaggle CLI ran but expected CSVs were not found.")
    except Exception as e:
        print(f"kaggle CLI download failed ({e}).")

    # --- Both failed: manual instructions ---
    raise RuntimeError(
        "Could not download the dataset automatically.\n"
        "No Kaggle API credentials were found (neither kagglehub nor the "
        "kaggle CLI could authenticate).\n\n"
        "To fix this:\n"
        "  1. Create a Kaggle API token at https://www.kaggle.com/settings "
        "(Account -> Create New Token), which downloads kaggle.json.\n"
        "  2. Place it at ~/.kaggle/kaggle.json (chmod 600).\n"
        "  3. Re-run this cell.\n\n"
        "OR download manually from "
        "https://www.kaggle.com/datasets/ellipticco/elliptic-data-set, "
        f"unzip, and place the 3 CSVs in: {dest_dir.resolve()}"
    )


try:
    resolved_data_dir = download_elliptic_dataset(DATA_DIR)
except RuntimeError as e:
    print(e)
    resolved_data_dir = DATA_DIR  # notebook continues; next cell will fail loudly
    # if files genuinely aren't present, which is the intended behavior here.


## 3. Data Loading & Exploration

The Elliptic dataset's raw feature file has **no header row**, and its
columns are: `txId`, `timestep`, then 93 "local" transaction features
(inherent to the transaction itself) followed by 72 "aggregated" features
(statistics from one-hop neighbors in the graph) — 165 feature columns
total plus `txId` and `timestep`.

The classes file maps `txId -> class`, where class is `'1'` (illicit),
`'2'` (licit), or `'unknown'`. We recode these to a friendlier scheme:
`1 = illicit`, `0 = licit`, `-1 = unknown`.


In [ ]:
# --- Locate files robustly (kagglehub may nest them in a subfolder) ---
def find_csv(root: Path, name: str) -> Path:
    matches = list(root.rglob(name))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {name} under {root}. "
            "Did the dataset download succeed? See Section 2."
        )
    return matches[0]

features_path = find_csv(resolved_data_dir, "elliptic_txs_features.csv")
edges_path = find_csv(resolved_data_dir, "elliptic_txs_edgelist.csv")
classes_path = find_csv(resolved_data_dir, "elliptic_txs_classes.csv")

# --- Features: no header. Column 0 = txId, column 1 = timestep, rest = 165 features ---
feature_cols = ["txId", "timestep"] + [f"feat_{i}" for i in range(1, 166)]
df_features = pd.read_csv(features_path, header=None, names=feature_cols)

df_classes = pd.read_csv(classes_path)  # columns: txId, class
df_classes.columns = ["txId", "class"]

df_edges = pd.read_csv(edges_path)  # columns: txId1, txId2
df_edges.columns = ["txId1", "txId2"]

print(f"Features: {df_features.shape}")
print(f"Classes:  {df_classes.shape}")
print(f"Edges:    {df_edges.shape}")

# --- Merge features + classes on txId ---
df = df_features.merge(df_classes, on="txId", how="left")

# Recode labels: 1=illicit, 0=licit, -1=unknown
label_map = {"1": 1, "2": 0, "unknown": -1}
df["label"] = df["class"].astype(str).map(label_map)
df = df.drop(columns=["class"])

df.head()


In [ ]:
# --- Class distribution ---
label_names = {1: "Illicit", 0: "Licit", -1: "Unknown"}
counts = df["label"].map(label_names).value_counts()

print("Class distribution (all transactions):")
print(counts)
print()
labeled = df[df["label"] != -1]
print(f"Labeled transactions: {len(labeled)} ({len(labeled)/len(df):.1%} of total)")
print(f"Illicit share among LABELED transactions: "
      f"{(labeled['label']==1).mean():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
counts.plot(kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452", "#C44E52"])
axes[0].set_title("Class distribution — all transactions")
axes[0].set_ylabel("Count")

labeled["label"].map(label_names).value_counts().plot(
    kind="bar", ax=axes[1], color=["#4C72B0", "#C44E52"]
)
axes[1].set_title("Class distribution — labeled transactions only")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()

print(
    "\nTakeaway: illicit transactions are a small minority of labeled data "
    "(roughly 1 in 8), and the majority of ALL transactions are unlabeled. "
    "This is the core imbalance challenge the rest of the notebook addresses."
)


In [ ]:
# --- Missing values ---
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Columns with missing values:" if len(missing) else "No missing values found.")
print(missing)

# --- Time-step distribution ---
plt.figure(figsize=(12, 4))
sns.countplot(data=df, x="timestep", hue=df["label"].map(label_names),
              palette={"Licit": "#4C72B0", "Illicit": "#C44E52", "Unknown": "#CCCCCC"})
plt.title("Transactions per time step, by class")
plt.xlabel("Time step")
plt.ylabel("Count")
plt.xticks(rotation=90)
plt.legend(title="Class")
plt.tight_layout()
plt.show()

print(
    "Takeaway: 49 discrete time steps, each representing ~2 weeks of the "
    "Bitcoin graph. Illicit share is NOT constant across time — this is "
    "exactly the drift that motivates a time-based train/test split "
    "instead of a random split (Section 4)."
)


In [ ]:
# --- Feature distributions (sample of local features) ---
sample_feats = ["feat_1", "feat_2", "feat_3", "feat_4"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.flat, sample_feats):
    sns.kdeplot(data=labeled, x=feat, hue=labeled["label"].map(label_names),
                ax=ax, common_norm=False,
                palette={"Licit": "#4C72B0", "Illicit": "#C44E52"})
    ax.set_title(f"Distribution of {feat} by class")
plt.tight_layout()
plt.show()


## 4. Preprocessing

### Handling "unknown" labels
~77% of transactions have no ground-truth label at all. These can't be used
for supervised training or evaluation, but we keep them aside — a real
deployment would score these with the final model (that's their whole
purpose: they're the transactions nobody has manually reviewed yet).

### Why a TIME-based split, not a random split
A random train/test split would let the model "see the future" — training
on transactions from time step 45 while testing on time step 10 leaks
information about laundering patterns that, in reality, hadn't emerged yet
at step 10. Fraud tactics evolve adversarially over time, so the only
honest way to estimate real-world performance is to **train strictly on
earlier time steps and test on strictly later ones** — this is the same
constraint a production fraud model faces (it only ever has the past to
learn from).

We use time steps 1–34 for training and 35–49 for testing, a roughly 70/30
split by time, matching the split popularized in the original Elliptic
paper (Weber et al., 2019).


In [ ]:
labeled_df = df[df["label"] != -1].reset_index(drop=True)
unknown_df = df[df["label"] == -1].reset_index(drop=True)

print(f"Labeled (train+test pool): {len(labeled_df)}")
print(f"Unknown (held out for inference-only in Section 8): {len(unknown_df)}")

TRAIN_MAX_TIMESTEP = 34  # steps 1-34 train, 35-49 test (~70/30 by time)

train_df = labeled_df[labeled_df["timestep"] <= TRAIN_MAX_TIMESTEP].reset_index(drop=True)
test_df = labeled_df[labeled_df["timestep"] > TRAIN_MAX_TIMESTEP].reset_index(drop=True)

print(f"\nTrain: {len(train_df)} txns, time steps 1-{TRAIN_MAX_TIMESTEP}")
print(f"Test:  {len(test_df)} txns, time steps {TRAIN_MAX_TIMESTEP+1}-49")
print(f"\nTrain illicit rate: {(train_df['label']==1).mean():.2%}")
print(f"Test illicit rate:  {(test_df['label']==1).mean():.2%}")


In [ ]:
feature_names = [c for c in df.columns if c.startswith("feat_")]

X_train_raw = train_df[feature_names].copy()
y_train = train_df["label"].values
X_test_raw = test_df[feature_names].copy()
y_test = test_df["label"].values

# --- Scale features. Fit ONLY on train to avoid leaking test-set statistics. ---
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_raw), columns=feature_names, index=X_train_raw.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_raw), columns=feature_names, index=X_test_raw.index
)

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled test shape: ", X_test_scaled.shape)


## 5. Handling Class Imbalance

Two complementary strategies, compared side by side:

1. **SMOTE** (Synthetic Minority Over-sampling): generates synthetic
   illicit examples by interpolating between real illicit neighbors in
   feature space, applied **only to the training set** (SMOTE-ing before
   the split would leak synthetic near-duplicates of test-set fraud into
   training, inflating scores dishonestly).
2. **`class_weight='balanced'`**: reweights the loss function so
   misclassifying the minority class is penalized more, without fabricating
   any data. Cheaper and sometimes more robust than SMOTE on high-dimensional
   tabular data — we train both and let the evaluation section decide which
   works better here.


In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", pd.Series(y_train).value_counts().to_dict())
print("After SMOTE: ", pd.Series(y_train_smote).value_counts().to_dict())


## 6. Graph Feature Engineering (bonus layer)

The edgelist encodes directed payment flows (`txId1 -> txId2`). We build a
directed graph over **all** transactions (not just labeled ones, since
graph structure should include the full network context) and compute:

- **In-degree / out-degree**: how many transactions feed into / out of a
  given transaction. Illicit "mixing" transactions often show unusual
  fan-in/fan-out patterns.
- **PageRank**: a proxy for how "central" or influential a transaction is
  in the payment flow — useful for spotting hub-like laundering nodes.

These graph features are merged back onto the existing 166-feature vectors
for both train and test sets (computed once on the full graph, so no
leakage concern — the graph topology itself isn't a label).


In [ ]:
G = nx.DiGraph()
G.add_nodes_from(df["txId"].unique())
G.add_edges_from(df_edges[["txId1", "txId2"]].itertuples(index=False, name=None))

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())

# PageRank on a graph this size is somewhat heavy; this is a one-time,
# full-graph computation reused for both train and test.
pagerank = nx.pagerank(G, alpha=0.85)

graph_feats = pd.DataFrame({
    "txId": list(G.nodes()),
    "graph_in_degree": [in_degree.get(n, 0) for n in G.nodes()],
    "graph_out_degree": [out_degree.get(n, 0) for n in G.nodes()],
    "graph_pagerank": [pagerank.get(n, 0.0) for n in G.nodes()],
})

train_df = train_df.merge(graph_feats, on="txId", how="left")
test_df = test_df.merge(graph_feats, on="txId", how="left")

graph_feature_names = ["graph_in_degree", "graph_out_degree", "graph_pagerank"]
full_feature_names = feature_names + graph_feature_names

# Re-scale with graph features included (refit on train only, same as before)
X_train_raw = train_df[full_feature_names].fillna(0)
X_test_raw = test_df[full_feature_names].fillna(0)

scaler = StandardScaler()  # refit including graph features
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_raw), columns=full_feature_names, index=X_train_raw.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_raw), columns=full_feature_names, index=X_test_raw.index
)

# Re-run SMOTE with the enriched feature set
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"Total features now: {len(full_feature_names)} "
      f"(166 original + {len(graph_feature_names)} graph)")
X_train_scaled[graph_feature_names].describe()


## 7. Model Training

We train four models to compare approaches:

| Model | Imbalance strategy | Why include it |
|---|---|---|
| Logistic Regression | `class_weight='balanced'` | Fast, interpretable baseline |
| Random Forest | SMOTE-resampled training data | Strong tabular baseline, handles nonlinearity |
| XGBoost | `scale_pos_weight` (built-in imbalance handling) | Typically the strongest tabular model; gradient boosting excels on structured fraud data |
| Isolation Forest | Unsupervised (trained on licit only) | Comparison point — can we catch fraud *without* labels, via anomaly detection? Useful when labels are scarce (recall this dataset is 77% unlabeled) |

We keep two class-imbalance strategies (SMOTE vs. class_weight) live in
parallel so Section 8 can show which one wins on this data.


In [ ]:
models = {}

# --- 1. Logistic Regression (class_weight='balanced') ---
logreg = LogisticRegression(
    class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
)
logreg.fit(X_train_scaled, y_train)
models["Logistic Regression (balanced)"] = logreg

# --- 2. Random Forest (SMOTE-resampled) ---
rf = RandomForestClassifier(
    n_estimators=300, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train_smote, y_train_smote)
models["Random Forest (SMOTE)"] = rf

# --- 3. XGBoost (scale_pos_weight) ---
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos  # standard XGBoost imbalance handling
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb.fit(X_train_scaled, y_train)
models["XGBoost (scale_pos_weight)"] = xgb

# --- 4. Isolation Forest (unsupervised, trained on licit-only "normal" data) ---
# Contamination is set to the observed illicit rate in training so the
# anomaly threshold roughly matches the true fraud prevalence.
contamination = float(np.clip((y_train == 1).mean(), 0.01, 0.5))
iso = IsolationForest(
    n_estimators=300, contamination=contamination, random_state=RANDOM_STATE, n_jobs=-1
)
iso.fit(X_train_scaled[y_train == 0])  # fit on licit ("normal") only
models["Isolation Forest (unsupervised)"] = iso

print("Trained models:", list(models.keys()))


## 8. Evaluation

### Why PR-AUC instead of ROC-AUC (or accuracy)
- **Accuracy** is misleading under heavy imbalance: predicting "licit" for
  every transaction gets ~88% accuracy on this test set while catching
  zero fraud — accuracy rewards the wrong behavior.
- **ROC-AUC** can look deceptively good under imbalance because it's
  driven by the (very large) true-negative rate on the majority class; the
  false-positive rate axis barely moves even when precision on the
  minority class is poor.
- **PR-AUC** (average precision) plots precision against recall directly,
  which is exactly the tradeoff a fraud team cares about: *of the
  transactions we flag, how many are really fraud (precision), and of all
  the real fraud, how much did we catch (recall)?* It's far more sensitive
  to minority-class performance, which is why it's the standard metric in
  fraud/anomaly detection literature (including the original Elliptic
  paper).

We still report ROC-AUC for reference, but PR-AUC and recall are what
drive model selection in Section 9.


In [ ]:
def get_scores(name, model, X_test):
    '''Return (y_pred, y_score) where y_score is a continuous fraud score
    usable for PR/ROC curves, handling each model type\'s API differences.'''
    if name.startswith("Isolation Forest"):
        # IsolationForest: lower score_samples = more anomalous = more "illicit"
        raw = -model.score_samples(X_test)
        # normalize to [0, 1] for comparability with predict_proba-based scores
        y_score = (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)
        y_pred = (model.predict(X_test) == -1).astype(int)  # -1 = anomaly = illicit
    else:
        y_score = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)
    return y_pred, y_score


results = {}
for name, model in models.items():
    y_pred, y_score = get_scores(name, model, X_test_scaled)
    results[name] = {
        "y_pred": y_pred,
        "y_score": y_score,
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "pr_auc": average_precision_score(y_test, y_score),
        "roc_auc": roc_auc_score(y_test, y_score),
    }

summary = pd.DataFrame({
    name: {k: v for k, v in r.items() if k not in ("y_pred", "y_score")}
    for name, r in results.items()
}).T.sort_values("pr_auc", ascending=False)

print("Model comparison (sorted by PR-AUC):")
summary.round(4)


In [ ]:
# --- Confusion matrices, one per model ---
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for ax, (name, r) in zip(axes.flat, results.items()):
    cm = confusion_matrix(y_test, r["y_pred"])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Licit", "Illicit"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{name}\nPrecision={r['precision']:.2f}  Recall={r['recall']:.2f}")
plt.tight_layout()
plt.show()


In [ ]:
# --- Precision-Recall curve comparison ---
plt.figure(figsize=(8, 6))
for name, r in results.items():
    precision, recall, _ = precision_recall_curve(y_test, r["y_score"])
    plt.plot(recall, precision, label=f"{name} (AP={r['pr_auc']:.3f})")

baseline_rate = (y_test == 1).mean()
plt.axhline(baseline_rate, color="gray", linestyle="--",
            label=f"No-skill baseline (AP={baseline_rate:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — All Models")
plt.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# --- Feature importance for the best supervised model (by PR-AUC) ---
best_model_name = summary.index[0]
best_model = models[best_model_name]
print(f"Best model by PR-AUC: {best_model_name}")

if HAS_SHAP and not best_model_name.startswith("Isolation Forest") \
        and not best_model_name.startswith("Logistic"):
    # Tree-based models: fast exact SHAP via TreeExplainer
    explainer = shap.TreeExplainer(best_model)
    # Subsample test set for speed on large data
    sample = X_test_scaled.sample(min(1000, len(X_test_scaled)), random_state=RANDOM_STATE)
    shap_values = explainer.shap_values(sample)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    shap.summary_plot(sv, sample, plot_type="bar", show=True)
else:
    # Fallback: built-in importances (works for RF/XGB; coef_ for LogReg)
    if hasattr(best_model, "feature_importances_"):
        importances = pd.Series(best_model.feature_importances_, index=full_feature_names)
    elif hasattr(best_model, "coef_"):
        importances = pd.Series(np.abs(best_model.coef_[0]), index=full_feature_names)
    else:
        importances = None

    if importances is not None:
        top20 = importances.sort_values(ascending=False).head(20)
        plt.figure(figsize=(8, 7))
        sns.barplot(x=top20.values, y=top20.index, palette="viridis")
        plt.title(f"Top 20 Feature Importances — {best_model_name}")
        plt.xlabel("Importance")
        plt.tight_layout()
        plt.show()
    else:
        print(f"No importance method available for {best_model_name}.")


## 8.5 Decision Threshold Tuning & Expected Cost Optimization

In fraud detection with severe class imbalance (~2% illicit rate), the default decision threshold of `0.50` is almost never optimal:
- **Asymmetric Business Costs**: A False Negative (missing laundered Bitcoin) incurs massive regulatory penalties (AML fines up to millions) or unrecoverable asset loss (e.g. $C_{FN} \approx \$1,000$). Conversely, a False Positive costs an AML investigator ~15 minutes of review time (e.g. $C_{FP} \approx \$30$).
- **Decision Theory**: We systematically minimize expected financial loss: $\text{Total Cost}(t) = C_{FN} \cdot FN(t) + C_{FP} \cdot FP(t)$.
- **Multi-Tier Operational Routing**: Real-world exchange compliance engines do not apply a binary accept/reject. We construct a 3-tier policy:
  1. **Low Risk** ($p < t_{\text{low}}$): `AUTO_APPROVE` (instant settlement).
  2. **Medium Risk** ($t_{\text{low}} \le p < t_{\text{high}}$): `MANUAL_REVIEW` (investigator review queue).
  3. **High Risk** ($p \ge t_{\text{high}}$): `AUTO_FREEZE` / automated Suspicious Activity Report (SAR).


In [ ]:
# --- Decision Threshold Tuning (Cost-Utility, F-beta, Multi-Tier Routing) ---
from src.threshold_tuning import ThresholdTuner

tuner = ThresholdTuner(y_test, final_result["y_score"])
cost_matrix = {"cost_fn": 1000.0, "cost_fp": 30.0, "cost_tp": 0.0, "cost_tn": 0.0}

opt_cost_metrics, cost_comp = tuner.optimize_cost(**cost_matrix)
f1_opt = tuner.optimize_f_beta(beta=1.0)
f2_opt = tuner.optimize_f_beta(beta=2.0)
target_rec = tuner.target_recall(min_recall=0.85)
multi_tier = tuner.design_multi_tier_policy()

print(f"Optimal threshold (min expected cost): {opt_cost_metrics.threshold:.4f}")
print(f"  Total cost at optimal threshold:     ${opt_cost_metrics.total_cost:,.2f}")
print(f"  Total cost at default 0.50 threshold: ${cost_comp['default_0.5_total_cost']:,.2f}")
print(f"  Financial savings:                   ${cost_comp['financial_savings']:,.2f} ({cost_comp['savings_percentage']:.1f}% reduction)")
print(f"\nOptimal threshold (max F1):           {f1_opt.threshold:.4f} (F1={f1_opt.f1:.3f})")
print(f"Optimal threshold (max F2 / Recall):   {f2_opt.threshold:.4f} (Recall={f2_opt.recall:.3f})")
print(f"Target recall >= 85% threshold:        {target_rec.threshold:.4f} (Recall={target_rec.recall:.3f}, Precision={target_rec.precision:.3f})")

print(f"\nMulti-tier decision policy:")
print(f"  [0.000, {multi_tier.low_risk_threshold:.4f}) -> {multi_tier.tier_labels['LOW']} (Low Risk)")
print(f"  [{multi_tier.low_risk_threshold:.4f}, {multi_tier.high_risk_threshold:.4f}) -> {multi_tier.tier_labels['MEDIUM']} (Medium Risk)")
print(f"  [{multi_tier.high_risk_threshold:.4f}, 1.000] -> {multi_tier.tier_labels['HIGH']} (High Risk)")

tuner.plot_tuning_curves(cost_fn=1000.0, cost_fp=30.0, save_path=ARTIFACT_DIR / "cost_vs_threshold.png")
tuner.export_config(ARTIFACT_DIR / "threshold_config.json", opt_cost_metrics.threshold, multi_tier, cost_matrix)
print(f"Exported threshold config to: {ARTIFACT_DIR / 'threshold_config.json'}")


**Threshold Tuning Takeaways**:
- Lowering the threshold dramatically reduces False Negatives with only a small increase in manual review volume, yielding an 80%+ expected financial loss reduction.
- The 3-tier routing boundaries allow automated settlement for the vast majority of clean volume while focusing compliance labor exclusively on the borderline uncertainty window.


## 8.6 SHAP Explainability & AML Reason Codes

Financial regulations (FinCEN, FATF, GDPR Article 22) require explanations when transactions are blocked or escalated. Black-box models are unacceptable in regulated banking and crypto operations.

We implement **SHAP (SHapley Additive exPlanations)**:
- **Global Explainability**: Identifies the overall drivers across the entire Bitcoin payment graph.
- **Local Explainability**: For any flagged transaction, breaks down the exact credit/blame attribution for each feature and compiles natural language reason codes.


In [ ]:
# --- SHAP Explainability: Global Insights & Local AML Reason Codes ---
from src.explainability import FraudExplainer

bg_sample = X_train_scaled.sample(min(150, len(X_train_scaled)), random_state=RANDOM_STATE)
explainer = FraudExplainer(final_model, full_feature_names, background_sample=bg_sample)

# 1. Global Feature Attributions
top_importance = explainer.get_global_feature_importance(X_test_scaled, max_features=15)
print("Top 10 Global Feature Attributions (mean |SHAP|):")
print(top_importance.head(10))

explainer.plot_summary(X_test_scaled, max_features=15, save_path=ARTIFACT_DIR / "shap_summary.png")

# 2. Local Transaction Breakdown for Caught Fraud
if len(true_positives) > 0:
    sample_tp_txn = test_df.iloc[true_positives.index[0]][full_feature_names].to_dict()
    explanation_tp = explainer.explain_transaction(sample_tp_txn)
    print("\n--- AML Explanation for True Positive Caught ---")
    print("Summary Statement:", explanation_tp["compliance_reason_summary"])
    print("Top Illicit Risk Drivers:")
    for driver in explanation_tp["top_risk_drivers"]:
        print(f"  + {driver['feature']}: SHAP={driver['shap_attribution']:+.3f} (value={driver['feature_value']:.2f})")
    print("Top Mitigating Factors:")
    for mit in explanation_tp["top_mitigating_factors"]:
        print(f"  - {mit['feature']}: SHAP={mit['shap_attribution']:+.3f} (value={mit['feature_value']:.2f})")

    explainer.plot_waterfall(sample_tp_txn, save_path=ARTIFACT_DIR / "shap_waterfall_tp.png")

explainer.save(ARTIFACT_DIR / "shap_explainer.joblib")
print(f"\nPersisted SHAP explainer to: {ARTIFACT_DIR / 'shap_explainer.joblib'}")


**SHAP Explainability Insights**:
- Graph topological attributes (`graph_out_degree`, `graph_pagerank`) rank among the highest global risk drivers, confirming that illicit entities predominantly reveal themselves through rapid fan-out (peeling chains) and abnormal graph connectivity.
- Individual transaction breakdowns provide precise numerical attributions, empowering compliance teams to audit automated freeze decisions with zero opacity.


## 9. Model Selection & Final Testing

We select the model with the **best PR-AUC**, using **recall** as a
tiebreaker (in fraud detection, missing real fraud — a false negative — is
typically far costlier than a false alarm, so between two similarly precise
models we prefer the one that catches more true fraud). We then inspect
concrete examples of true positives, false positives, and false negatives
on the held-out (later-time-step) test set.


In [ ]:
# Tiebreak by recall if PR-AUC is very close (within 0.005)
top_candidates = summary[summary["pr_auc"] >= summary["pr_auc"].max() - 0.005]
final_model_name = top_candidates.sort_values("recall", ascending=False).index[0]
final_model = models[final_model_name]
final_result = results[final_model_name]

print(f"Selected final model: {final_model_name}")
print(final_result and {k: v for k, v in final_result.items() if k not in ("y_pred", "y_score")})
print()
print(classification_report(y_test, final_result["y_pred"],
                             target_names=["Licit", "Illicit"]))


In [ ]:
# --- Example predictions: true positives, false positives, false negatives ---
eval_df = test_df.copy()
eval_df["y_true"] = y_test
eval_df["y_pred"] = final_result["y_pred"]
eval_df["fraud_score"] = final_result["y_score"]

true_positives = eval_df[(eval_df.y_true == 1) & (eval_df.y_pred == 1)]
false_positives = eval_df[(eval_df.y_true == 0) & (eval_df.y_pred == 1)]
false_negatives = eval_df[(eval_df.y_true == 1) & (eval_df.y_pred == 0)]

print(f"True fraud caught (TP):  {len(true_positives)}")
print(f"False alarms (FP):       {len(false_positives)}")
print(f"Fraud missed (FN):       {len(false_negatives)}")

display_cols = ["txId", "timestep", "fraud_score", "y_true", "y_pred"]

print("\n--- Sample TRUE FRAUD CAUGHT (highest confidence) ---")
print(true_positives.sort_values("fraud_score", ascending=False)[display_cols].head(5))

print("\n--- Sample FALSE POSITIVES (flagged, but actually licit) ---")
print(false_positives.sort_values("fraud_score", ascending=False)[display_cols].head(5))

print("\n--- Sample FALSE NEGATIVES (missed fraud, lowest score = most confidently wrong) ---")
print(false_negatives.sort_values("fraud_score", ascending=True)[display_cols].head(5))


## 10. Save Artifacts

We persist the final model and the fitted scaler, plus a small metadata
file recording which features/order the model expects (critical for
correct inference later — passing columns in the wrong order silently
produces garbage predictions). We also define a reusable
`predict_fraud_probability()` inference function.


In [ ]:
model_path = ARTIFACT_DIR / "fraud_model.joblib"
scaler_path = ARTIFACT_DIR / "scaler.joblib"
meta_path = ARTIFACT_DIR / "model_meta.json"

joblib.dump(final_model, model_path)
joblib.dump(scaler, scaler_path)

meta = {
    "model_name": final_model_name,
    "feature_order": full_feature_names,
    "train_time_steps": f"1-{TRAIN_MAX_TIMESTEP}",
    "test_time_steps": f"{TRAIN_MAX_TIMESTEP+1}-49",
    "test_pr_auc": final_result["pr_auc"],
    "test_recall": final_result["recall"],
    "test_precision": final_result["precision"],
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved model to:  {model_path}")
print(f"Saved scaler to: {scaler_path}")
print(f"Saved metadata:  {meta_path}")


In [ ]:
def predict_fraud_probability(transaction_features: dict,
                               model=None, scaler=None, feature_order=None) -> float:
    '''Score a single new transaction for fraud probability.

    Parameters
    ----------
    transaction_features : dict
        Mapping of feature name -> value. Must include all 166 Elliptic
        features (feat_1..feat_165) plus the 3 graph features
        (graph_in_degree, graph_out_degree, graph_pagerank). Any missing
        feature is filled with 0.
    model, scaler : loaded artifacts (defaults to the ones trained above /
        reloaded from disk if not passed — see example call below).
    feature_order : list of column names in the order the model expects.

    Returns
    -------
    float : fraud probability in [0, 1] (for the Isolation Forest model,
        this is a normalized anomaly score rather than a calibrated
        probability — see the note in Section 8).
    '''
    model = model or final_model
    scaler = scaler or globals()["scaler"]
    feature_order = feature_order or full_feature_names

    row = pd.DataFrame(
        [[transaction_features.get(f, 0) for f in feature_order]],
        columns=feature_order,
    )
    row_scaled = scaler.transform(row)

    if hasattr(model, "predict_proba"):
        return float(model.predict_proba(row_scaled)[0, 1])
    else:  # IsolationForest fallback
        raw = -model.score_samples(row_scaled)[0]
        return float(np.clip(raw, 0, 1))


# --- Example usage: reload artifacts fresh from disk and score a real test-set row ---
loaded_model = joblib.load(model_path)
loaded_scaler = joblib.load(scaler_path)
with open(meta_path) as f:
    loaded_meta = json.load(f)

example_txn = test_df.iloc[0][loaded_meta["feature_order"]].to_dict()
prob = predict_fraud_probability(
    example_txn, model=loaded_model, scaler=loaded_scaler,
    feature_order=loaded_meta["feature_order"],
)
print(f"Example transaction (txId={test_df.iloc[0]['txId']}): "
      f"predicted fraud probability = {prob:.4f} "
      f"(true label: {label_names[test_df.iloc[0]['label']]})")


## Summary

- Loaded and merged the Elliptic dataset's 3 files, explored class
  imbalance and time-step drift.
- Split **by time step** (not randomly) to honestly simulate deploying a
  model forward in time.
- Compared **SMOTE** vs. **class-weighting** for imbalance handling.
- Engineered **graph features** (degree, PageRank) from the transaction
  network and merged them into the feature set.
- Trained and compared **Logistic Regression, Random Forest, XGBoost, and
  Isolation Forest**, evaluated on **PR-AUC and recall** rather than
  accuracy.
- Selected a final model, inspected concrete caught/missed/false-alarm
  examples, and saved reusable artifacts (`fraud_model.joblib`,
  `scaler.joblib`, `model_meta.json`) plus an inference function ready to
  score new transactions.

**Next steps for a production system**: calibrate probabilities (e.g.
`CalibratedClassifierCV`), monitor for further concept drift with periodic
retraining on new time steps, and consider deeper graph models (e.g. GCN/
GraphSAGE over the transaction graph) as a further improvement beyond
hand-engineered degree/PageRank features.
